In [1]:
import pandas as pd
import numpy as np
import re 
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
#load dataset

df = pd.read_excel(r"C:\Users\DELL\Documents\ppppp\phishing_dataset.xlsx")
print(df.head())

   id                      subject               sender  \
0   1               Monthly Report    admin@company.com   
1   2  Salary Payment Notification       hr@company.com   
2   3              Order Delivered       sales@shop.com   
3   4  Salary Payment Notification       hr@company.com   
4   5        Team Meeting Reminder  manager@company.com   

                                          email_text  has_link  \
0           Please find attached the monthly report.         1   
1        Your salary has been credited successfully.         1   
2  Your order has been delivered. Thank you for s...         0   
3        Your salary has been credited successfully.         0   
4            Meeting scheduled for tomorrow at 10am.         0   

   has_attachment  urgency_flag       label  
0               0             0  legitimate  
1               1             0  legitimate  
2               1             0  legitimate  
3               1             0  legitimate  
4               0 

In [3]:
print("dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

dataset shape: (800, 8)

Columns:
['id', 'subject', 'sender', 'email_text', 'has_link', 'has_attachment', 'urgency_flag', 'label']


In [4]:
#missing values
print(df.isnull().sum())

id                0
subject           0
sender            0
email_text        0
has_link          0
has_attachment    0
urgency_flag      0
label             0
dtype: int64


In [5]:
df["combined_text"] = (
    df["subject"].astype(str) + " " +
    df["sender"].astype(str) + " " +
    df["email_text"].astype(str)
)

df[["subject", "sender", "email_text", "combined_text"]].head()

,subject,sender,email_text,combined_text
0,Monthly Report,admin@company.com,Please find attached the monthly report.,Monthly Report admin@company.com Please find a...
1,Salary Payment Notification,hr@company.com,Your salary has been credited successfully.,Salary Payment Notification hr@company.com You...
2,Order Delivered,sales@shop.com,Your order has been delivered. Thank you for s...,Order Delivered sales@shop.com Your order has ...
3,Salary Payment Notification,hr@company.com,Your salary has been credited successfully.,Salary Payment Notification hr@company.com You...
4,Team Meeting Reminder,manager@company.com,Meeting scheduled for tomorrow at 10am.,Team Meeting Reminder manager@company.com Meet...


In [6]:
# Create one final text field for the email
df["combined_text"] = (
    df["subject"].astype(str) + " " +
    df["sender"].astype(str) + " " +
    df["email_text"].astype(str)
)

# numeric form

df["label_num"] = df["label"].map({"phishing": 1, "legitimate": 0})

df_model = df[[
    "combined_text",
    "has_link",
    "has_attachment",
    "urgency_flag",
    "label_num"
]]

df_model.head()

,combined_text,has_link,has_attachment,urgency_flag,label_num
0,Monthly Report admin@company.com Please find a...,1,0,0,0
1,Salary Payment Notification hr@company.com You...,1,1,0,0
2,Order Delivered sales@shop.com Your order has ...,0,1,0,0
3,Salary Payment Notification hr@company.com You...,0,1,0,0
4,Team Meeting Reminder manager@company.com Meet...,0,0,0,0


In [7]:
df_model["label_num"].value_counts()


label_num
1    414
0    386
Name: count, dtype: int64

In [8]:
df["label"].value_counts()

label
phishing      414
legitimate    386
Name: count, dtype: int64

In [9]:
# Text input for TF-IDF
X_text = df_model["combined_text"]

# Extra metadata / security indicators
X_meta = df_model[["has_link", "has_attachment", "urgency_flag"]]

# Target labels
y = df_model["label_num"]

print("X_text sample:")
print(X_text.head())

print("\nX_meta sample:")
print(X_meta.head())

print("\ny sample:")
print(y.head())

X_text sample:
0    Monthly Report admin@company.com Please find a...
1    Salary Payment Notification hr@company.com You...
2    Order Delivered sales@shop.com Your order has ...
3    Salary Payment Notification hr@company.com You...
4    Team Meeting Reminder manager@company.com Meet...
Name: combined_text, dtype: object

X_meta sample:
   has_link  has_attachment  urgency_flag
0         1               0             0
1         1               1             0
2         0               1             0
3         0               1             0
4         0               0             0

y sample:
0    0
1    0
2    0
3    0
4    0
Name: label_num, dtype: int64


In [10]:
# Split text, metadata, and labels into train and test sets
X_text_train, X_text_test, X_meta_train, X_meta_test, y_train, y_test = train_test_split(
    X_text,
    X_meta,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training text samples:", X_text_train.shape)
print("Testing text samples:", X_text_test.shape)

print("\nTraining metadata shape:", X_meta_train.shape)
print("Testing metadata shape:", X_meta_test.shape)

print("\nTraining labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

Training text samples: (640,)
Testing text samples: (160,)

Training metadata shape: (640, 3)
Testing metadata shape: (160, 3)

Training labels shape: (640,)
Testing labels shape: (160,)


In [11]:
# Convert text into TF-IDF features
'''vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_text_train_tfidf = vectorizer.fit_transform(X_text_train)
X_text_test_tfidf = vectorizer.transform(X_text_test)

print("TF-IDF train shape:", X_text_train_tfidf.shape)
print("TF-IDF test shape:", X_text_test_tfidf.shape)'''

'vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)\n\nX_text_train_tfidf = vectorizer.fit_transform(X_text_train)\nX_text_test_tfidf = vectorizer.transform(X_text_test)\n\nprint("TF-IDF train shape:", X_text_train_tfidf.shape)\nprint("TF-IDF test shape:", X_text_test_tfidf.shape)'

In [12]:
# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

# Learn vocabulary from training data
X_train_tfidf = tfidf.fit_transform(X_text_train)

# Transform testing data using the same vocabulary
X_test_tfidf = tfidf.transform(X_text_test)

print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Testing TF-IDF Shape:", X_test_tfidf.shape)

Training TF-IDF Shape: (640, 62)
Testing TF-IDF Shape: (160, 62)


In [13]:
from scipy.sparse import hstack

# Combine TF-IDF text features with metadata features
X_train_final = hstack([X_train_tfidf, X_meta_train.values])
X_test_final = hstack([X_test_tfidf, X_meta_test.values])

print("Final Training Shape:", X_train_final.shape)
print("Final Testing Shape:", X_test_final.shape)

Final Training Shape: (640, 65)
Final Testing Shape: (160, 65)


In [14]:
from sklearn.linear_model import LogisticRegression

# Create and train the ML model
ml_model = LogisticRegression(max_iter=1000)
ml_model.fit(X_train_final, y_train)

print("Machine Learning model trained successfully!")

Machine Learning model trained successfully!


In [15]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on test data
y_pred = ml_model.predict(X_test_final)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        77
           1       1.00      1.00      1.00        83

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160


Confusion Matrix:

[[77  0]
 [ 0 83]]


In [16]:
# Get prediction probabilities
y_proba = ml_model.predict_proba(X_test_final)

# Probability of phishing class (class 1)
phishing_confidence = y_proba[:, 1]

print("First 10 phishing confidence scores:")
print(phishing_confidence[:10])

First 10 phishing confidence scores:
[0.98600876 0.00844926 0.02734069 0.98521778 0.98399899 0.98399899
 0.98577652 0.00844926 0.00696619 0.98577652]


In [17]:
# Create readable predictions
results_df = pd.DataFrame({
    "email_text": X_text_test.values,
    "actual_label": y_test.values,
    "predicted_label": y_pred,
    "phishing_confidence": phishing_confidence
})

# Convert labels into readable text
results_df["actual_label"] = results_df["actual_label"].map({1: "phishing", 0: "legitimate"})
results_df["predicted_label"] = results_df["predicted_label"].map({1: "phishing", 0: "legitimate"})

# Show first 10 results
results_df.head(10)

,email_text,actual_label,predicted_label,phishing_confidence
0,Account suspended alert support@bank-secure.co...,phishing,phishing,0.986009
1,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.008449
2,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.027341
3,Account suspended alert support@bank-secure.co...,phishing,phishing,0.985218
4,Password Expiry Notice it-support@securemail.c...,phishing,phishing,0.983999
5,Password Expiry Notice it-support@securemail.c...,phishing,phishing,0.983999
6,BVN Update Required noreply@fakebank.com Urgen...,phishing,phishing,0.985777
7,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.008449
8,Salary Payment Notification hr@company.com You...,legitimate,legitimate,0.006966
9,BVN Update Required noreply@fakebank.com Urgen...,phishing,phishing,0.985777


In [18]:
import joblib
import os

# Create models folder if it does not exist
os.makedirs("models", exist_ok=True)

# Save trained ML model
joblib.dump(ml_model, "models/ml_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

print("ML model and TF-IDF vectorizer saved successfully!")

ML model and TF-IDF vectorizer saved successfully!


In [19]:
# cell 15

#starting Deep Learning & Machine learning


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizer settings
max_words = 5000
max_len = 200

# Create tokenizer
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_text_train)

# Convert text into sequences
X_train_seq = tokenizer.texts_to_sequences(X_text_train)
X_test_seq = tokenizer.texts_to_sequences(X_text_test)

# Pad sequences so all emails have same length
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

print("Training padded shape:", X_train_pad.shape)
print("Testing padded shape:", X_test_pad.shape)

Training padded shape: (640, 200)
Testing padded shape: (160, 200)


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

# Build Deep Learning model
dl_model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

# Compile model
dl_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Show model summary
dl_model.summary()

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Train Deep Learning model
history = dl_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.5352 - loss: 0.6945 - val_accuracy: 0.4766 - val_loss: 0.6909
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.5586 - loss: 0.6881 - val_accuracy: 0.4766 - val_loss: 0.6934
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5488 - loss: 0.6804 - val_accuracy: 0.4766 - val_loss: 0.6775
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6348 - loss: 0.6742 - val_accuracy: 0.4766 - val_loss: 0.6699
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6973 - loss: 0.6573 - val_accuracy: 1.0000 - val_loss: 0.6388
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7793 - loss: 0.6223 - val_accuracy: 1.0000 - val_loss: 0.6022
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8555 - loss: 0.5846 - val_accuracy: 1.0000 - val_loss: 0.5412
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9668 - loss: 0.5080 - val_accuracy: 1.0000 - v

In [23]:
#Evaluate Deep Learning model on test set
dl_loss, dl_acc = dl_model.evaluate(X_test_pad, y_test, verbose=0)
print("Deep Learning Test Accuracy:", dl_acc)
print("Deep Learning Test Loss:", dl_loss)

Deep Learning Test Accuracy: 1.0
Deep Learning Test Loss: 0.2734614908695221


In [24]:
import numpy as np

#probablity prediction  from deep learning model
dl_probs = dl_model.predict(X_test_pad)

#convert probabilities into class labels
dl_pred = (dl_probs > 0.5).astype("int32").flatten()

print("First 10 DL phishing probabilities:")
print(dl_probs[:10].flatten())

print("\nFirst 10 DL predicted labels:")
print(dl_pred[:10])


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step  
First 10 DL phishing probabilities:
[0.86563516 0.321556   0.321556   0.86563516 0.8592675  0.8592675
 0.86086214 0.321556   0.36145836 0.86086214]

First 10 DL predicted labels:
[1 0 0 1 1 1 1 0 0 1]


In [25]:
#create readable DL results table
dl_results_df = pd.DataFrame({
    "email_text": X_text_test.values,
    "actual_label": y_test.values,
    "dl_predicted_label":dl_pred,
    "dl_phishing_confidence":dl_probs.flatten()
})

#convert labels into readable text
dl_results_df["actual_label"]= dl_results_df["actual_label"].map({1: "phishing", 0: "legitimate"})
dl_results_df["dl_predicted_label"]= dl_results_df["dl_predicted_label"].map({1: "phishing", 0: "legitimate"})

#show first 10 rows
dl_results_df.head(10)

,email_text,actual_label,dl_predicted_label,dl_phishing_confidence
0,Account suspended alert support@bank-secure.co...,phishing,phishing,0.865635
1,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.321556
2,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.321556
3,Account suspended alert support@bank-secure.co...,phishing,phishing,0.865635
4,Password Expiry Notice it-support@securemail.c...,phishing,phishing,0.859267
5,Password Expiry Notice it-support@securemail.c...,phishing,phishing,0.859267
6,BVN Update Required noreply@fakebank.com Urgen...,phishing,phishing,0.860862
7,Monthly Report admin@company.com Please find a...,legitimate,legitimate,0.321556
8,Salary Payment Notification hr@company.com You...,legitimate,legitimate,0.361458
9,BVN Update Required noreply@fakebank.com Urgen...,phishing,phishing,0.860862


In [26]:
# save Deep Learning MOdel
os.makedirs("models", exist_ok=True)
dl_model.save("models/dl_model.keras")

#save tokenizer 
joblib.dump(tokenizer,"models/tokenizer.pkl")

print("Deep learning model and tokenizer saved successfully")

Deep learning model and tokenizer saved successfully


In [27]:
import re
import joblib
import numpy as np
from scipy.sparse import hstack
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [28]:
#load trained artifacts
ml_model=joblib.load("models/ml_model.pkl")
tfidf=joblib.load("models/tfidf_vectorizer.pkl")
dl_model=load_model("models/dl_model.keras")
tokenizer = joblib.load("models/tokenizer.pkl")

MAX_LEN =200

def clean_text(text):
    """
    clean email text:
    -lower case
    -remove URLs
    -remove special characters
    -remove extra spaces
    """
    if text is None:
        return""
        text = str(text).lower()
        text=re.sub(r"http\s+|www\s+", "", text)  
        text=re.sub(r"[a-zA-Z0-9\s", "",text)
        text=re.sub(r"\s+", "", text).strip()

# Feature preparation

def prepare_email_input(subject, sender, email_text, has_links, has_attachment, urgency_flag):
    """
    Prepare input excatly like training time.
    Returns:
      combined_text_clean
      meta_array
    """
    subject = ""if subject is None else str(subject)
    sender = ""if sender is None else str(sender)
    email_text = ""if email_text is None else str(email_text)

    combined_text = subject + " " + sender + "" + email_text
    combined_text_clean = clean_text(combined_text)

    
    

In [29]:
import re
import joblib
import numpy as np
from scipy.sparse import hstack
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


# =========================
# LOAD TRAINED ARTIFACTS
# =========================
ml_model = joblib.load("models/ml_model.pkl")
tfidf = joblib.load("models/tfidf_vectorizer.pkl")
dl_model = load_model("models/dl_model.keras")
tokenizer = joblib.load("models/tokenizer.pkl")

# Same value as training notebook
MAX_LEN = 200


# =========================
# TEXT PREPROCESSING
# =========================
def clean_text(text):
    """
    Clean email text:
    - lower case
    - remove URLs
    - remove special characters
    - remove extra spaces
    """
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove urls
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)   # keep only letters/numbers/spaces
    text = re.sub(r"\s+", " ", text).strip()      # remove extra spaces
    return text


# =========================
# FEATURE PREPARATION
# =========================
def prepare_email_input(subject, sender, email_text, has_link, has_attachment, urgency_flag):
    """
    Prepare input exactly like training time.
    Returns:
        combined_text_clean
        meta_array
    """
    subject = "" if subject is None else str(subject)
    sender = "" if sender is None else str(sender)
    email_text = "" if email_text is None else str(email_text)

    # Combine text same as notebook
    combined_text = subject + " " + sender + " " + email_text
    combined_text_clean = clean_text(combined_text)

    # Metadata array shape: (1, 3)
    meta_array = np.array([[int(has_link), int(has_attachment), int(urgency_flag)]])

    return combined_text_clean, meta_array


# =========================
# ML PREDICTION
# =========================
def predict_ml(subject, sender, email_text, has_link, has_attachment, urgency_flag):
    """
    Predict using Logistic Regression model
    """
    combined_text_clean, meta_array = prepare_email_input(
        subject, sender, email_text, has_link, has_attachment, urgency_flag
    )

    # TF-IDF transform
    text_tfidf = tfidf.transform([combined_text_clean])

    # Combine text features + metadata
    final_features = hstack([text_tfidf, meta_array])

    # Predict label
    pred_label = ml_model.predict(final_features)[0]

    # Predict phishing probability
    pred_proba = ml_model.predict_proba(final_features)[0][1]

    label_name = "phishing" if pred_label == 1 else "legitimate"

    return {
        "model": "Machine Learning (Logistic Regression)",
        "label": int(pred_label),
        "label_name": label_name,
        "phishing_confidence": round(float(pred_proba) * 100, 2)
    }


# =========================
# DL PREDICTION
# =========================
def predict_dl(subject, sender, email_text, has_link, has_attachment, urgency_flag):
    """
    Predict using Deep Learning model
    Note:
    Current DL model uses only text input (same as notebook training).
    Metadata is accepted for API consistency but not used inside DL model right now.
    """
    combined_text_clean, meta_array = prepare_email_input(
        subject, sender, email_text, has_link, has_attachment, urgency_flag
    )

    # Convert text to sequence
    seq = tokenizer.texts_to_sequences([combined_text_clean])

    # Pad sequence
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

    # Predict probability
    dl_prob = float(dl_model.predict(padded, verbose=0)[0][0])

    # Convert to label
    pred_label = 1 if dl_prob > 0.5 else 0
    label_name = "phishing" if pred_label == 1 else "legitimate"

    return {
        "model": "Deep Learning (TensorFlow/Keras)",
        "label": int(pred_label),
        "label_name": label_name,
        "phishing_confidence": round(dl_prob * 100, 2)
    }


# =========================
# FINAL HYBRID PREDICTION
# =========================
def predict_email(subject, sender, email_text, has_link, has_attachment, urgency_flag):
    """
    Run both ML and DL models and return a combined response.
    """
    ml_result = predict_ml(subject, sender, email_text, has_link, has_attachment, urgency_flag)
    dl_result = predict_dl(subject, sender, email_text, has_link, has_attachment, urgency_flag)

    # Final decision strategy:
    # 1) If either model says phishing -> final phishing
    # 2) Final confidence = max(ml_confidence, dl_confidence)
    if ml_result["label"] == 1 or dl_result["label"] == 1:
        final_label = 1
        final_label_name = "phishing"
    else:
        final_label = 0
        final_label_name = "legitimate"

    final_confidence = max(
        ml_result["phishing_confidence"],
        dl_result["phishing_confidence"]
    )

    return {
        "final_prediction": final_label_name,
        "final_label": final_label,
        "final_confidence": final_confidence,
        "ml_result": ml_result,
        "dl_result": dl_result
    }


# =========================
# TEST RUN (optional)
# =========================
if __name__ == "__main__":
    sample = predict_email(
        subject="Urgent Account Verification",
        sender="support@fakebank.com",
        email_text="Your account will be suspended. Click here to verify your login immediately.",
        has_link=1,
        has_attachment=0,
        urgency_flag=1
    )

    print("Sample Prediction Result:")
    print(sample)

Sample Prediction Result:
{'final_prediction': 'phishing', 'final_label': 1, 'final_confidence': 98.44, 'ml_result': {'model': 'Machine Learning (Logistic Regression)', 'label': 1, 'label_name': 'phishing', 'phishing_confidence': 98.44}, 'dl_result': {'model': 'Deep Learning (TensorFlow/Keras)', 'label': 1, 'label_name': 'phishing', 'phishing_confidence': 85.36}}


In [3]:
import imaplib
import email
from email.header import decode_header
import re

# ---------------------------------------------------
# GMAIL CONFIG
# ---------------------------------------------------
IMAP_SERVER = "imap.gmail.com"
IMAP_PORT = 993


# ---------------------------------------------------
# HELPER: decode email subject safely
# ---------------------------------------------------
def decode_mime_text(text):
    """
    Decode MIME encoded email subject/text.
    Example: =?UTF-8?...?=
    """
    if not text:
        return ""

    decoded_parts = decode_header(text)
    final_text = ""

    for part, encoding in decoded_parts:
        if isinstance(part, bytes):
            try:
                final_text += part.decode(encoding or "utf-8", errors="ignore")
            except:
                final_text += part.decode("utf-8", errors="ignore")
        else:
            final_text += str(part)

    return final_text


# ---------------------------------------------------
# HELPER: extract plain text body from email
# ---------------------------------------------------
def extract_email_body(msg):
    """
    Extract plain text body from email message.
    If multipart, loop through parts and pick text/plain.
    """
    body = ""

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))

            # Skip attachments
            if "attachment" in content_disposition.lower():
                continue

            if content_type == "text/plain":
                try:
                    body = part.get_payload(decode=True).decode(errors="ignore")
                    return body
                except:
                    pass
    else:
        try:
            body = msg.get_payload(decode=True).decode(errors="ignore")
        except:
            body = ""

    return body


# ---------------------------------------------------
# HELPER: check if email contains link
# ---------------------------------------------------
def has_link_in_text(text):
    if not text:
        return 0

    url_pattern = r"(https?://\S+|www\.\S+)"
    return 1 if re.search(url_pattern, text, re.IGNORECASE) else 0


# ---------------------------------------------------
# HELPER: detect urgency words
# ---------------------------------------------------
def has_urgency_words(text):
    if not text:
        return 0

    urgency_keywords = [
        "urgent", "immediately", "verify", "suspended",
        "action required", "click now", "password reset",
        "security alert", "limited time", "confirm now"
    ]

    text_lower = text.lower()
    for word in urgency_keywords:
        if word in text_lower:
            return 1
    return 0


# ---------------------------------------------------
# HELPER: check if email has attachment
# ---------------------------------------------------
def has_attachment(msg):
    """
    Return 1 if email has any attachment, else 0
    """
    for part in msg.walk():
        content_disposition = str(part.get("Content-Disposition"))
        if "attachment" in content_disposition.lower():
            return 1
    return 0


# ---------------------------------------------------
# MAIN FUNCTION: fetch latest Gmail emails
# ---------------------------------------------------
def fetch_latest_gmail_emails(gmail_address, app_password, max_emails=5):
    """
    Connect to Gmail inbox using IMAP and fetch latest emails.

    Returns a list of dictionaries:
    [
        {
            "subject": "...",
            "sender": "...",
            "email_text": "...",
            "has_link": 0/1,
            "has_attachment": 0/1,
            "urgency_flag": 0/1
        }
    ]
    """

    mail = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
    mail.login(gmail_address, app_password)
    mail.select("inbox")

    status, messages = mail.search(None, "ALL")
    email_ids = messages[0].split()

    # Get latest emails only
    latest_email_ids = email_ids[-max_emails:]

    email_data_list = []

    for email_id in reversed(latest_email_ids):
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        raw_email = msg_data[0][1]
        msg = email.message_from_bytes(raw_email)

        subject = decode_mime_text(msg.get("Subject"))
        sender = msg.get("From", "")
        body = extract_email_body(msg)

        link_flag = has_link_in_text(body)
        attachment_flag = has_attachment(msg)
        urgency_flag = has_urgency_words(subject + " " + body)

        email_data = {
            "subject": subject,
            "sender": sender,
            "email_text": body,
            "has_link": link_flag,
            "has_attachment": attachment_flag,
            "urgency_flag": urgency_flag
        }

        email_data_list.append(email_data)

    mail.logout()
    return email_data_list


# ---------------------------------------------------
# TEST MODE
# ---------------------------------------------------
if __name__ == "__main__":
    # IMPORTANT:
    # Replace with your Gmail and Gmail App Password
    gmail_address = "abc@gmail.com"
    app_password = "abcd efgh ijkl mnop"

    emails = fetch_latest_gmail_emails(gmail_address, app_password, max_emails=3)

    print("Fetched Emails:\n")
    for i, item in enumerate(emails, start=1):
        print(f"Email {i}")
        print("Subject:", item["subject"])
        print("Sender:", item["sender"])
        print("Has Link:", item["has_link"])
        print("Has Attachment:", item["has_attachment"])
        print("Urgency Flag:", item["urgency_flag"])
        print("Body Preview:", item["email_text"][:200])
        print("-" * 80)

error: b'[AUTHENTICATIONFAILED] Invalid credentials (Failure)'

In [2]:
def fetch_latest_gmail_emails(gmail_address, app_password, max_emails=5):
    """
    Fetch latest emails from Gmail using IMAP with proper error handling
    """
    import imaplib
    import email
    
    # Gmail IMAP settings
    IMAP_SERVER = "imap.gmail.com"
    IMAP_PORT = 993
    
    try:
        # Connect to Gmail with error handling
        mail = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
        
        # Login with proper error handling
        mail.login(gmail_address, app_password)
        mail.select("inbox")

        status, messages = mail.search(None, "ALL")
        email_ids = messages[0].split()

        # Get latest emails only
        latest_email_ids = email_ids[-max_emails:]

        email_data_list = []

        for email_id in reversed(latest_email_ids):
            status, msg_data = mail.fetch(email_id, "(RFC822)")
            raw_email = msg_data[0][1]
            msg = email.message_from_bytes(raw_email)

            subject = decode_mime_text(msg.get("Subject"))
            sender = msg.get("From", "")
            body = extract_email_body(msg)

            link_flag = has_link_in_text(body)
            attachment_flag = has_attachment(msg)
            urgency_flag = has_urgency_words(subject + " " + body)

            email_data = {
                "subject": subject,
                "sender": sender,
                "email_text": body,
                "has_link": link_flag,
                "has_attachment": attachment_flag,
                "urgency_flag": urgency_flag
            }

            email_data_list.append(email_data)

        mail.logout()
        return email_data_list
        
    except imaplib.IMAP4.error as e:
        # Handle IMAP authentication errors
        print(f"IMAP Error: {e}")
        print("Please check:")
        print("1. Your email address is correct")
        print("2. You're using a Gmail App Password (not regular password)")
        print("3. 2-Factor Authentication is enabled on your Gmail")
        print("4. The App Password is the 16-character code from Gmail")
        return []
    except Exception as e:
        # Handle other errors
        print(f"Unexpected error: {e}")
        return []


# ---------------------------------------------------
# TEST MODE
# ---------------------------------------------------
if __name__ == "__main__":
    # IMPORTANT: Replace with your actual credentials
    # Use your Gmail App Password, NOT your regular Gmail password
    gmail_address = "your_email@gmail.com"  # Replace with your actual Gmail
    app_password = "abcd efgh ijkl mnop"    # Replace with your 16-char App Password
    
    print("Instructions to get Gmail App Password:")
    print("1. Go to your Google Account settings")
    print("2. Enable 2-Factor Authentication")
    print("3. Go to Security > App passwords")
    print("4. Generate a new app password for 'Mail'")
    print("5. Use that 16-character password here")
    print("-" * 50)

    emails = fetch_latest_gmail_emails(gmail_address, app_password, max_emails=3)

    if emails:
        print("Fetched Emails:\n")
        for i, item in enumerate(emails, start=1):
            print(f"Email {i}")
            print("Subject:", item["subject"])
            print("Sender:", item["sender"])
            print("Has Link:", item["has_link"])
            print("Has Attachment:", item["has_attachment"])
            print("Urgency Flag:", item["urgency_flag"])
            print("Body Preview:", item["email_text"][:200])
            print("-" * 80)
    else:
        print("No emails fetched. Please check your credentials.")

Instructions to get Gmail App Password:
1. Go to your Google Account settings
2. Enable 2-Factor Authentication
3. Go to Security > App passwords
4. Generate a new app password for 'Mail'
5. Use that 16-character password here
--------------------------------------------------
IMAP Error: b'[AUTHENTICATIONFAILED] Invalid credentials (Failure)'
Please check:
1. Your email address is correct
2. You're using a Gmail App Password (not regular password)
3. 2-Factor Authentication is enabled on your Gmail
4. The App Password is the 16-character code from Gmail
No emails fetched. Please check your credentials.
